In [36]:
import pandas as pd

# Load the dataset
try:
    df_original = pd.read_csv('/content/retail_store_sales (1).csv')
    print('Dataset loaded successfully.')
except FileNotFoundError:
    print('Error: The file retail_store_sales (1).csv was not found.')
    df_original = pd.DataFrame()

# Keep a copy of the original dataframe for comparison later
df = df_original.copy()

Dataset loaded successfully.


### 1. Inspect the Dataset

In [37]:
# Identify the number of rows and columns
original_rows, original_cols = df.shape
print(f"Original number of rows: {original_rows}")
print(f"Original number of columns: {original_cols}")

# Display all column names and their data types
print("\nColumn names and data types:")
display(df.info())

Original number of rows: 12575
Original number of columns: 11

Column names and data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              11362 non-null  object 
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(8)
memory usage: 1.1+ MB


None

In [38]:
# Check every column for missing, null, blank, or invalid values
print("\nMissing values count per column:")
missing_values_count = df.isnull().sum()
display(missing_values_count[missing_values_count > 0])

print("\nTotal missing values in the dataset:")
total_missing_values = df.isnull().sum().sum()
print(total_missing_values)


Missing values count per column:


,0
Item,1213
Price Per Unit,609
Quantity,604
Total Spent,604
Discount Applied,4199



Total missing values in the dataset:
7229


In [39]:
# Identify and count duplicate rows
duplicate_rows = df.duplicated()
num_duplicate_rows = duplicate_rows.sum()
print(f"\nNumber of duplicate rows found: {num_duplicate_rows}")

if num_duplicate_rows > 0:
    print("\nExample of duplicate rows (first 5 duplicates):")
    display(df[duplicate_rows].head())


Number of duplicate rows found: 0


Now that we have inspected the dataset, let's proceed to handle the missing values and clean up the 'Discount Applied' column based on the specified requirements.

### 2. Handle Missing Values

### 3. Special Requirement — `Discount Applied` Column

I will address the `Discount Applied` column first, as it has specific rules, and its values might help in inferring other missing values.

In [40]:
# Convert 'Discount Applied' to True/False
# First, standardize common representations to a temporary boolean form
def clean_discount_applied(value):
    if pd.isna(value) or str(value).strip() == '':
        return None # Keep NaNs/empty as None for further processing
    elif str(value).lower() in ['true', 'yes', 'y', '1']:
        return True
    elif str(value).lower() in ['false', 'no', 'n', '0']:
        return False
    return value # Return original value if it's an unrecognized format

df['Discount Applied'] = df['Discount Applied'].apply(clean_discount_applied)

# Check if there are still any non-boolean values or NaNs in 'Discount Applied'
non_boolean_discount_values = df[~df['Discount Applied'].isin([True, False])]['Discount Applied'].unique()
if len(non_boolean_discount_values) > 0:
    print(f"Warning: 'Discount Applied' still contains non-boolean values after initial cleaning: {non_boolean_discount_values}. These will be handled in subsequent steps.")
else:
    print("All 'Discount Applied' values are True, False, or None after initial cleaning.")


print("\n'Discount Applied' column after initial cleaning:")
display(df['Discount Applied'].value_counts(dropna=False))


'Discount Applied' column after initial cleaning:


,count
Discount Applied,
True,4219
None,4199
False,4157


In [41]:
# Initialize a dictionary to store how missing values were handled for documentation
missing_values_handled_log = {}
initial_missing_counts = df.isnull().sum()
missing_values_handled_log['initial_counts_before_detailed_handling'] = initial_missing_counts[initial_missing_counts > 0].to_dict()

# --- Step 1: Ensure numerical columns are truly numeric ---
# (Already float64, but this is a good safety step to handle potential non-numeric strings)
for col in ['Price Per Unit', 'Quantity', 'Total Spent']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# --- Step 2: Refine Discount Applied based on numerical relationships ---
# This step is crucial for inferring Discount Applied where it's None and to validate existing flags.
# Create a potential 'undiscounted total' for comparison. Handle division by zero for Quantity and Price Per Unit for robustness.
df['Undiscounted Total'] = df['Price Per Unit'] * df['Quantity']

# --- Step 2.1: Infer 'Discount Applied' status where it's None ---

# Case A: Discount Applied is None, Total Spent is missing, but Undiscounted Total can be calculated.
# This implies no discount was applied if Total Spent should be equal to Undiscounted Total.
mask_none_discount_total_missing_no_discount = (
    df['Discount Applied'].isna() &
    df['Total Spent'].isna() &
    df['Undiscounted Total'].notna()
)
num_inferred_no_discount_calc_total = mask_none_discount_total_missing_no_discount.sum()
if num_inferred_no_discount_calc_total > 0:
    df.loc[mask_none_discount_total_missing_no_discount, 'Discount Applied'] = False
    df.loc[mask_none_discount_total_missing_no_discount, 'Total Spent'] = df.loc[mask_none_discount_total_missing_no_discount, 'Undiscounted Total']
    missing_values_handled_log['Discount Applied'] = missing_values_handled_log.get('Discount Applied', []) + [f"{num_inferred_no_discount_calc_total} values inferred as False and Total Spent calculated from (Price Per Unit * Quantity) where Discount Applied was None and Total Spent was missing."]
    missing_values_handled_log['Total Spent'] = missing_values_handled_log.get('Total Spent', []) + [f"{num_inferred_no_discount_calc_total} values calculated from Price Per Unit * Quantity where Discount Applied was None and Total Spent was missing."]


# Case B: Discount Applied is None, and Total Spent is present. Compare Total Spent with Undiscounted Total.
mask_none_discount_total_present_undiscounted_notna = (
    df['Discount Applied'].isna() &
    df['Total Spent'].notna() &
    df['Undiscounted Total'].notna()
)
tolerance = 1e-6 # For float comparison

# Infer False if Total Spent is approximately equal to Undiscounted Total
mask_no_discount_inferred = mask_none_discount_total_present_undiscounted_notna & (abs(df['Total Spent'] - df['Undiscounted Total']) < tolerance)
num_inferred_no_discount = mask_no_discount_inferred.sum()
if num_inferred_no_discount > 0:
    df.loc[mask_no_discount_inferred, 'Discount Applied'] = False
    missing_values_handled_log['Discount Applied'] = missing_values_handled_log.get('Discount Applied', []) + [f"{num_inferred_no_discount} values inferred as False where (Price Per Unit * Quantity) == Total Spent and Discount Applied was None."]

# Infer True if Total Spent is less than Undiscounted Total (implies a discount)
mask_discount_inferred = mask_none_discount_total_present_undiscounted_notna & (df['Total Spent'] < df['Undiscounted Total'])
num_inferred_discount = mask_discount_inferred.sum()
if num_inferred_discount > 0:
    df.loc[mask_discount_inferred, 'Discount Applied'] = True
    missing_values_handled_log['Discount Applied'] = missing_values_handled_log.get('Discount Applied', []) + [f"{num_inferred_discount} values inferred as True where Total Spent < (Price Per Unit * Quantity) and Discount Applied was None."]

# Remaining None in 'Discount Applied' (e.g., Total Spent > Undiscounted Total, or PPU/Quantity missing for Undiscounted Total calc).
# For ambiguity (Total Spent > Undiscounted Total) or insufficient info, we'll default to False for 'Discount Applied'.
num_remaining_none_discount = df['Discount Applied'].isna().sum()
if num_remaining_none_discount > 0:
    df.loc[df['Discount Applied'].isna(), 'Discount Applied'] = False
    missing_values_handled_log['Discount Applied'] = missing_values_handled_log.get('Discount Applied', []) + [f"{num_remaining_none_discount} remaining None values for Discount Applied defaulted to False (due to ambiguity or insufficient information for precise inference)."]

# Ensure 'Discount Applied' is of boolean type
df['Discount Applied'] = df['Discount Applied'].astype(bool)


# --- Step 3: Impute numerical columns using logical calculations where possible ---
# This step uses the refined 'Discount Applied' column.

# Fill missing 'Total Spent' (if PPU and Quantity are available and no discount)
mask_total_spent_missing_no_discount = df['Total Spent'].isna() & df['Price Per Unit'].notna() & df['Quantity'].notna() & (df['Discount Applied'] == False)
num_calculated_total_spent = mask_total_spent_missing_no_discount.sum()
if num_calculated_total_spent > 0:
    df.loc[mask_total_spent_missing_no_discount, 'Total Spent'] = df.loc[mask_total_spent_missing_no_discount, 'Price Per Unit'] * df.loc[mask_total_spent_missing_no_discount, 'Quantity']
    missing_values_handled_log['Total Spent'] = missing_values_handled_log.get('Total Spent', []) + [f"{num_calculated_total_spent} values calculated from Price Per Unit * Quantity where Discount Applied is False."]

# Fill missing 'Price Per Unit' (if Total Spent and Quantity are available and no discount, and Quantity is not zero)
mask_price_missing_no_discount = df['Price Per Unit'].isna() & df['Total Spent'].notna() & df['Quantity'].notna() & (df['Discount Applied'] == False) & (df['Quantity'] != 0)
num_calculated_price = mask_price_missing_no_discount.sum()
if num_calculated_price > 0:
    df.loc[mask_price_missing_no_discount, 'Price Per Unit'] = df.loc[mask_price_missing_no_discount, 'Total Spent'] / df.loc[mask_price_missing_no_discount, 'Quantity']
    missing_values_handled_log['Price Per Unit'] = missing_values_handled_log.get('Price Per Unit', []) + [f"{num_calculated_price} values calculated from Total Spent / Quantity where Discount Applied is False and Quantity != 0."]

# Fill missing 'Quantity' (if Total Spent and Price Per Unit are available and no discount, and Price Per Unit is not zero)
mask_quantity_missing_no_discount = df['Quantity'].isna() & df['Total Spent'].notna() & df['Price Per Unit'].notna() & (df['Discount Applied'] == False) & (df['Price Per Unit'] != 0)
num_calculated_quantity = mask_quantity_missing_no_discount.sum()
if num_calculated_quantity > 0:
    df.loc[mask_quantity_missing_no_discount, 'Quantity'] = df.loc[mask_quantity_missing_no_discount, 'Total Spent'] / df.loc[mask_quantity_missing_no_discount, 'Price Per Unit']
    missing_values_handled_log['Quantity'] = missing_values_handled_log.get('Quantity', []) + [f"{num_calculated_quantity} values calculated from Total Spent / Price Per Unit where Discount Applied is False and Price Per Unit != 0."]

# --- Step 4: Median imputation for remaining numerical missing values ---
# This is applied if a value still remains NaN after logical inference.

for col in ['Price Per Unit', 'Quantity', 'Total Spent']:
    # Impute for rows where Discount Applied is True
    median_true = df.loc[df['Discount Applied'] == True, col].median()
    mask_true_missing = df[col].isna() & (df['Discount Applied'] == True)
    num_imputed_true = mask_true_missing.sum()
    if num_imputed_true > 0:
        df.loc[mask_true_missing, col] = median_true
        missing_values_handled_log[col] = missing_values_handled_log.get(col, []) + [f"{num_imputed_true} values imputed with median ({median_true}) for Discount Applied = True."]

    # Impute for rows where Discount Applied is False
    median_false = df.loc[df['Discount Applied'] == False, col].median()
    mask_false_missing = df[col].isna() & (df['Discount Applied'] == False)
    num_imputed_false = mask_false_missing.sum()
    if num_imputed_false > 0:
        df.loc[mask_false_missing, col] = median_false
        missing_values_handled_log[col] = missing_values_handled_log.get(col, []) + [f"{num_imputed_false} values imputed with median ({median_false}) for Discount Applied = False."]

# Drop the temporary 'Undiscounted Total' column
df = df.drop(columns=['Undiscounted Total'])


# --- Step 5: Handle missing 'Item' values (categorical) ---
if 'Item' in df.columns:
    num_missing_item_initial = df['Item'].isna().sum()
    if num_missing_item_initial > 0:
        # Fill missing 'Item' based on 'Category' mode
        # Use transform to fill based on group mode, then fill remaining with overall mode
        df['Item'] = df.groupby('Category')['Item'].transform(lambda x: x.mode()[0] if not x.mode().empty else None)

        # Fill any remaining missing 'Item' (e.g., if 'Category' was also missing, or no mode for category) with overall mode
        overall_item_mode = df['Item'].mode()[0] if not df['Item'].mode().empty else 'Unknown Item'
        num_remaining_item_missing = df['Item'].isna().sum()
        if num_remaining_item_missing > 0:
            df.loc[df['Item'].isna(), 'Item'] = overall_item_mode
            missing_values_handled_log['Item'] = missing_values_handled_log.get('Item', []) + [f"{num_missing_item_initial - num_remaining_item_missing} values imputed using category mode, and {num_remaining_item_missing} values imputed with overall mode ({overall_item_mode})."]
        else:
            missing_values_handled_log['Item'] = missing_values_handled_log.get('Item', []) + [f"{num_missing_item_initial} values imputed using category mode."]
    else:
        missing_values_handled_log['Item'] = ['No missing values for Item.']
else:
    missing_values_handled_log['Item'] = ['Item column not found.']


# Verification of missing values after handling
print("\nMissing values count after handling:")
final_missing_counts = df.isnull().sum()
display(final_missing_counts[final_missing_counts > 0])

print("\nMissing values handling summary:")
for col, methods in missing_values_handled_log.items():
    print(f"- {col}:")
    if isinstance(methods, list):
        for method in methods:
            print(f"  - {method}")
    else:
        print(f"  - {methods}")



Missing values count after handling:


,0



Missing values handling summary:
- initial_counts_before_detailed_handling:
  - {'Item': 1213, 'Price Per Unit': 609, 'Quantity': 604, 'Total Spent': 604, 'Discount Applied': 4199}
- Discount Applied:
  - 3783 values inferred as False where (Price Per Unit * Quantity) == Total Spent and Discount Applied was None.
  - 416 remaining None values for Discount Applied defaulted to False (due to ambiguity or insufficient information for precise inference).
- Price Per Unit:
  - 391 values calculated from Total Spent / Quantity where Discount Applied is False and Quantity != 0.
  - 218 values imputed with median (23.0) for Discount Applied = True.
- Quantity:
  - 200 values imputed with median (6.0) for Discount Applied = True.
  - 404 values imputed with median (6.0) for Discount Applied = False.
- Total Spent:
  - 200 values imputed with median (109.5) for Discount Applied = True.
  - 404 values imputed with median (108.5) for Discount Applied = False.
- Item:
  - 1213 values imputed using

### 4. Remove Duplicates

In [42]:
# Identify and count duplicate rows (again, after potential changes from missing value handling)
duplicate_rows_after_cleaning = df.duplicated()
num_duplicate_rows_after_cleaning = duplicate_rows_after_cleaning.sum()

print(f"\nNumber of duplicate rows found before removal: {num_duplicate_rows_after_cleaning}")

# Remove duplicate rows while retaining the first valid occurrence
df.drop_duplicates(inplace=True)

num_removed_duplicates = num_duplicate_rows_after_cleaning - df.duplicated().sum()
print(f"Number of duplicate rows removed: {num_removed_duplicates}")

# Update log
missing_values_handled_log['Duplicates'] = [f"{num_duplicate_rows_after_cleaning} duplicate rows detected, {num_removed_duplicates} removed."]


Number of duplicate rows found before removal: 0
Number of duplicate rows removed: 0


### 5. Clean Data Formatting

In [43]:
# Remove unnecessary leading and trailing spaces from text fields
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
missing_values_handled_log['Formatting'] = missing_values_handled_log.get('Formatting', []) + ['Removed leading/trailing spaces from object columns.']

# Standardize inconsistent capitalization and spelling where appropriate.
# For 'Category', 'Payment Method', 'Location', converting to title case for consistency.
for col in ['Category', 'Payment Method', 'Location', 'Item']:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: x.title() if isinstance(x, str) else x)
missing_values_handled_log['Formatting'].append('Standardized capitalization for Category, Payment Method, Location, and Item columns to Title Case.')

# Ensure numerical columns contain valid numerical values (already handled during missing value imputation)
# Ensure dates are stored in a consistent format
if 'Transaction Date' in df.columns:
    # Convert to datetime, coercing errors will turn invalid dates into NaT
    df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

    # Handle NaT values if any (from invalid date strings)
    num_invalid_dates = df['Transaction Date'].isna().sum()
    if num_invalid_dates > 0:
        # Option 1: Impute with mode/median date. Option 2: Remove rows. Option 3: Fill with a placeholder (e.g., min date)
        # Given instruction to not use generic placeholders and attempt to determine, let's try mode.
        date_mode = df['Transaction Date'].mode()[0]
        df.loc[df['Transaction Date'].isna(), 'Transaction Date'] = date_mode
        missing_values_handled_log['Transaction Date'] = missing_values_handled_log.get('Transaction Date', []) + [f"{num_invalid_dates} invalid date values converted to NaT and then imputed with the mode date ({date_mode.strftime('%Y-%m-%d')})."]
    else:
        missing_values_handled_log['Transaction Date'] = missing_values_handled_log.get('Transaction Date', []) + ['No invalid dates found. Dates formatted to datetime.']

missing_values_handled_log['Formatting'].append('Converted Transaction Date to consistent datetime format.')

print("Data formatting cleanup completed.")

Data formatting cleanup completed.


### 6. Validate the Cleaned Dataset

In [44]:
print("\n--- Final Validation Check ---")

# 1. Verify that all missing values have been appropriately handled.
final_missing_check = df.isnull().sum()
missing_cols_after_cleaning = final_missing_check[final_missing_check > 0]
if missing_cols_after_cleaning.empty:
    print("Validation 1: PASSED - No missing values remain in the dataset.")
else:
    print("Validation 1: FAILED - Missing values still present in columns:")
    display(missing_cols_after_cleaning)
    missing_values_handled_log['Final Missing Check'] = missing_values_handled_log.get('Final Missing Check', []) + ['FAILED - Missing values still present after cleaning.']

# 2. Verify that there are zero duplicate rows.
num_duplicate_rows_final = df.duplicated().sum()
if num_duplicate_rows_final == 0:
    print("Validation 2: PASSED - Zero duplicate rows remain.")
else:
    print(f"Validation 2: FAILED - {num_duplicate_rows_final} duplicate rows still exist.")
    missing_values_handled_log['Final Duplicate Check'] = missing_values_handled_log.get('Final Duplicate Check', []) + [f'FAILED - {num_duplicate_rows_final} duplicate rows remain.']

# 3. Verify that every column has the correct data type.
# Check numerical columns
numerical_cols = ['Price Per Unit', 'Quantity', 'Total Spent']
num_type_check_passed = True
for col in numerical_cols:
    if col in df.columns and not pd.api.types.is_numeric_dtype(df[col]):
        print(f"Validation 3: FAILED - Column '{col}' is not a numeric type (is {df[col].dtype}).")
        num_type_check_passed = False
        missing_values_handled_log['Final Data Type Check'] = missing_values_handled_log.get('Final Data Type Check', []) + [f'FAILED - Column {col} is not numeric.']

# Check 'Discount Applied' for boolean type
if 'Discount Applied' in df.columns and not pd.api.types.is_bool_dtype(df['Discount Applied']):
    print(f"Validation 3: FAILED - Column 'Discount Applied' is not boolean type (is {df['Discount Applied'].dtype}).")
    num_type_check_passed = False
    missing_values_handled_log['Final Data Type Check'] = missing_values_handled_log.get('Final Data Type Check', []) + [f'FAILED - Column Discount Applied is not boolean.']

# Check 'Transaction Date' for datetime type
if 'Transaction Date' in df.columns and not pd.api.types.is_datetime64_any_dtype(df['Transaction Date']):
    print(f"Validation 3: FAILED - Column 'Transaction Date' is not datetime type (is {df['Transaction Date'].dtype}).")
    num_type_check_passed = False
    missing_values_handled_log['Final Data Type Check'] = missing_values_handled_log.get('Final Data Type Check', []) + [f'FAILED - Column Transaction Date is not datetime.']

if num_type_check_passed:
    print("Validation 3: PASSED - Data types appear correct for key columns.")
    missing_values_handled_log['Final Data Type Check'] = missing_values_handled_log.get('Final Data Type Check', []) + ['PASSED - Data types correct.']

# 4. Verify that the 'Discount Applied' column contains ONLY True and False.
if 'Discount Applied' in df.columns:
    unique_discount_values = df['Discount Applied'].unique()
    if set(unique_discount_values).issubset({True, False}):
        print("Validation 4: PASSED - 'Discount Applied' column contains only True/False.")
    else:
        print(f"Validation 4: FAILED - 'Discount Applied' column contains unexpected values: {unique_discount_values}")
        missing_values_handled_log['Final Discount Applied Check'] = missing_values_handled_log.get('Final Discount Applied Check', []) + [f'FAILED - Contains unexpected values: {unique_discount_values}.']
else:
    print("Validation 4: FAILED - 'Discount Applied' column not found.")
    missing_values_handled_log['Final Discount Applied Check'] = missing_values_handled_log.get('Final Discount Applied Check', []) + ['FAILED - Column not found.']

# 5. Check for invalid or inconsistent values again (spot check categorical columns).
# This is already somewhat covered by type checks and title casing, but a final look.
print("Validation 5: Performing spot checks on categorical columns...")
# For simplicity, just display value counts to visually inspect for inconsistencies.
for col in ['Category', 'Payment Method', 'Location']:
    if col in df.columns:
        print(f"  Unique values for '{col}':")
        display(df[col].value_counts())
print("Validation 5: PASSED - Spot checks performed. Manual review of value_counts may be needed for full consistency.")

final_rows, final_cols = df.shape
missing_values_handled_log['Final Dataset Dimensions'] = {'rows': final_cols, 'columns': final_cols}



--- Final Validation Check ---
Validation 1: PASSED - No missing values remain in the dataset.
Validation 2: PASSED - Zero duplicate rows remain.
Validation 3: PASSED - Data types appear correct for key columns.
Validation 4: PASSED - 'Discount Applied' column contains only True/False.
Validation 5: Performing spot checks on categorical columns...
  Unique values for 'Category':


,count
Category,
Electric Household Essentials,1591
Furniture,1591
Food,1588
Milk Products,1584
Butchers,1568
Beverages,1567
Computers And Electric Accessories,1558
Patisserie,1528


  Unique values for 'Payment Method':


,count
Payment Method,
Cash,4310
Digital Wallet,4144
Credit Card,4121


  Unique values for 'Location':


,count
Location,
Online,6354
In-Store,6221


Validation 5: PASSED - Spot checks performed. Manual review of value_counts may be needed for full consistency.


### 7. Export the Dataset

In [45]:
output_filename = 'CLEANED_DATASET.csv'
df.to_csv(output_filename, index=False)
print(f"\nCleaned dataset exported successfully to '{output_filename}'.")
missing_values_handled_log['Export Location'] = output_filename


Cleaned dataset exported successfully to 'CLEANED_DATASET.csv'.


### 8. Provide a Cleaning Summary

In [46]:
print("\n--- Data Cleaning Summary Report ---")
print(f"Original number of rows: {original_rows}")
print(f"Original number of columns: {original_cols}")

print("\nTotal missing values detected (initial):")
for col, count in missing_values_handled_log['initial_counts_before_detailed_handling'].items():
    print(f"- {col}: {count}")

print("\nMissing values found and methods used to replace in each column:")
for col, methods in missing_values_handled_log.items():
    if col not in ['initial_counts_before_detailed_handling', 'Duplicates', 'Formatting', 'Final Missing Check', 'Final Duplicate Check', 'Final Data Type Check', 'Final Discount Applied Check', 'Export Location', 'Final Dataset Dimensions']:
        print(f"- {col}:")
        if isinstance(methods, list):
            for method in methods:
                print(f"  - {method}")
        else:
            print(f"  - {methods}")

print("\nDuplicate Rows Handling:")
if 'Duplicates' in missing_values_handled_log:
    for entry in missing_values_handled_log['Duplicates']:
        print(f"- {entry}")

print("\nOther Corrections Performed (Formatting):")
if 'Formatting' in missing_values_handled_log:
    for entry in missing_values_handled_log['Formatting']:
        print(f"- {entry}")

print("\nFinal Dataset State:")
print(f"Final number of rows: {df.shape[0]}")
print(f"Final number of columns: {df.shape[1]}")

print(f"Confirmation: No duplicate rows remain - {num_duplicate_rows_final == 0}")
print(f"Confirmation: 'Discount Applied' column contains only True and False - {set(df['Discount Applied'].unique()).issubset({True, False})}")
print(f"Location of exported file: {output_filename}")

print("\n--- Cleaning Summary End ---")


--- Data Cleaning Summary Report ---
Original number of rows: 12575
Original number of columns: 11

Total missing values detected (initial):
- Item: 1213
- Price Per Unit: 609
- Quantity: 604
- Total Spent: 604
- Discount Applied: 4199

Missing values found and methods used to replace in each column:
- Discount Applied:
  - 3783 values inferred as False where (Price Per Unit * Quantity) == Total Spent and Discount Applied was None.
  - 416 remaining None values for Discount Applied defaulted to False (due to ambiguity or insufficient information for precise inference).
- Price Per Unit:
  - 391 values calculated from Total Spent / Quantity where Discount Applied is False and Quantity != 0.
  - 218 values imputed with median (23.0) for Discount Applied = True.
- Quantity:
  - 200 values imputed with median (6.0) for Discount Applied = True.
  - 404 values imputed with median (6.0) for Discount Applied = False.
- Total Spent:
  - 200 values imputed with median (109.5) for Discount Appli

In [47]:
# Initialize a dictionary to store how missing values were handled for documentation
missing_values_handled_log = {}
initial_missing_counts = df.isnull().sum()
missing_values_handled_log['initial_counts_before_detailed_handling'] = initial_missing_counts[initial_missing_counts > 0].to_dict()

# --- Step 1: Ensure numerical columns are truly numeric ---
# (Already float64, but this is a good safety step to handle potential non-numeric strings)
for col in ['Price Per Unit', 'Quantity', 'Total Spent']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# --- Step 2: Refine Discount Applied based on numerical relationships ---
# This step is crucial for inferring Discount Applied where it's None and to validate existing flags.
# Create a potential 'undiscounted total' for comparison. Handle division by zero for Quantity and Price Per Unit for robustness.
df['Undiscounted Total'] = df['Price Per Unit'] * df['Quantity']

# --- Step 2.1: Infer 'Discount Applied' status where it's None ---

# Case A: Discount Applied is None, Total Spent is missing, but Undiscounted Total can be calculated.
# This implies no discount was applied if Total Spent should be equal to Undiscounted Total.
mask_none_discount_total_missing_no_discount = (
    df['Discount Applied'].isna() &
    df['Total Spent'].isna() &
    df['Undiscounted Total'].notna()
)
num_inferred_no_discount_calc_total = mask_none_discount_total_missing_no_discount.sum()
if num_inferred_no_discount_calc_total > 0:
    df.loc[mask_none_discount_total_missing_no_discount, 'Discount Applied'] = False
    df.loc[mask_none_discount_total_missing_no_discount, 'Total Spent'] = df.loc[mask_none_discount_total_missing_no_discount, 'Undiscounted Total']
    missing_values_handled_log['Discount Applied'] = missing_values_handled_log.get('Discount Applied', []) + [f"{num_inferred_no_discount_calc_total} values inferred as False and Total Spent calculated from (Price Per Unit * Quantity) where Discount Applied was None and Total Spent was missing."]
    missing_values_handled_log['Total Spent'] = missing_values_handled_log.get('Total Spent', []) + [f"{num_inferred_no_discount_calc_total} values calculated from Price Per Unit * Quantity where Discount Applied was None and Total Spent was missing."]


# Case B: Discount Applied is None, and Total Spent is present. Compare Total Spent with Undiscounted Total.
mask_none_discount_total_present_undiscounted_notna = (
    df['Discount Applied'].isna() &
    df['Total Spent'].notna() &
    df['Undiscounted Total'].notna()
)
tolerance = 1e-6 # For float comparison

# Infer False if Total Spent is approximately equal to Undiscounted Total
mask_no_discount_inferred = mask_none_discount_total_present_undiscounted_notna & (abs(df['Total Spent'] - df['Undiscounted Total']) < tolerance)
num_inferred_no_discount = mask_no_discount_inferred.sum()
if num_inferred_no_discount > 0:
    df.loc[mask_no_discount_inferred, 'Discount Applied'] = False
    missing_values_handled_log['Discount Applied'] = missing_values_handled_log.get('Discount Applied', []) + [f"{num_inferred_no_discount} values inferred as False where (Price Per Unit * Quantity) == Total Spent and Discount Applied was None."]

# Infer True if Total Spent is less than Undiscounted Total (implies a discount)
mask_discount_inferred = mask_none_discount_total_present_undiscounted_notna & (df['Total Spent'] < df['Undiscounted Total'])
num_inferred_discount = mask_discount_inferred.sum()
if num_inferred_discount > 0:
    df.loc[mask_discount_inferred, 'Discount Applied'] = True
    missing_values_handled_log['Discount Applied'] = missing_values_handled_log.get('Discount Applied', []) + [f"{num_inferred_discount} values inferred as True where Total Spent < (Price Per Unit * Quantity) and Discount Applied was None."]

# Remaining None in 'Discount Applied' (e.g., Total Spent > Undiscounted Total, or PPU/Quantity missing for Undiscounted Total calc).
# For ambiguity (Total Spent > Undiscounted Total) or insufficient info, we'll default to False for 'Discount Applied'.
num_remaining_none_discount = df['Discount Applied'].isna().sum()
if num_remaining_none_discount > 0:
    df.loc[df['Discount Applied'].isna(), 'Discount Applied'] = False
    missing_values_handled_log['Discount Applied'] = missing_values_handled_log.get('Discount Applied', []) + [f"{num_remaining_none_discount} remaining None values for Discount Applied defaulted to False (due to ambiguity or insufficient information for precise inference)."]

# Ensure 'Discount Applied' is of boolean type
df['Discount Applied'] = df['Discount Applied'].astype(bool)


# --- Step 3: Impute numerical columns using logical calculations where possible ---
# This step uses the refined 'Discount Applied' column.

# Fill missing 'Total Spent' (if PPU and Quantity are available and no discount)
mask_total_spent_missing_no_discount = df['Total Spent'].isna() & df['Price Per Unit'].notna() & df['Quantity'].notna() & (df['Discount Applied'] == False)
num_calculated_total_spent = mask_total_spent_missing_no_discount.sum()
if num_calculated_total_spent > 0:
    df.loc[mask_total_spent_missing_no_discount, 'Total Spent'] = df.loc[mask_total_spent_missing_no_discount, 'Price Per Unit'] * df.loc[mask_total_spent_missing_no_discount, 'Quantity']
    missing_values_handled_log['Total Spent'] = missing_values_handled_log.get('Total Spent', []) + [f"{num_calculated_total_spent} values calculated from Price Per Unit * Quantity where Discount Applied is False."]

# Fill missing 'Price Per Unit' (if Total Spent and Quantity are available and no discount, and Quantity is not zero)
mask_price_missing_no_discount = df['Price Per Unit'].isna() & df['Total Spent'].notna() & df['Quantity'].notna() & (df['Discount Applied'] == False) & (df['Quantity'] != 0)
num_calculated_price = mask_price_missing_no_discount.sum()
if num_calculated_price > 0:
    df.loc[mask_price_missing_no_discount, 'Price Per Unit'] = df.loc[mask_price_missing_no_discount, 'Total Spent'] / df.loc[mask_price_missing_no_discount, 'Quantity']
    missing_values_handled_log['Price Per Unit'] = missing_values_handled_log.get('Price Per Unit', []) + [f"{num_calculated_price} values calculated from Total Spent / Quantity where Discount Applied is False and Quantity != 0."]

# Fill missing 'Quantity' (if Total Spent and Price Per Unit are available and no discount, and Price Per Unit is not zero)
mask_quantity_missing_no_discount = df['Quantity'].isna() & df['Total Spent'].notna() & df['Price Per Unit'].notna() & (df['Discount Applied'] == False) & (df['Price Per Unit'] != 0)
num_calculated_quantity = mask_quantity_missing_no_discount.sum()
if num_calculated_quantity > 0:
    df.loc[mask_quantity_missing_no_discount, 'Quantity'] = df.loc[mask_quantity_missing_no_discount, 'Total Spent'] / df.loc[mask_quantity_missing_no_discount, 'Price Per Unit']
    missing_values_handled_log['Quantity'] = missing_values_handled_log.get('Quantity', []) + [f"{num_calculated_quantity} values calculated from Total Spent / Price Per Unit where Discount Applied is False and Price Per Unit != 0."]

# --- Step 4: Median imputation for remaining numerical missing values ---
# This is applied if a value still remains NaN after logical inference.

for col in ['Price Per Unit', 'Quantity', 'Total Spent']:
    # Impute for rows where Discount Applied is True
    median_true = df.loc[df['Discount Applied'] == True, col].median()
    mask_true_missing = df[col].isna() & (df['Discount Applied'] == True)
    num_imputed_true = mask_true_missing.sum()
    if num_imputed_true > 0:
        df.loc[mask_true_missing, col] = median_true
        missing_values_handled_log[col] = missing_values_handled_log.get(col, []) + [f"{num_imputed_true} values imputed with median ({median_true}) for Discount Applied = True."]

    # Impute for rows where Discount Applied is False
    median_false = df.loc[df['Discount Applied'] == False, col].median()
    mask_false_missing = df[col].isna() & (df['Discount Applied'] == False)
    num_imputed_false = mask_false_missing.sum()
    if num_imputed_false > 0:
        df.loc[mask_false_missing, col] = median_false
        missing_values_handled_log[col] = missing_values_handled_log.get(col, []) + [f"{num_imputed_false} values imputed with median ({median_false}) for Discount Applied = False."]

# Drop the temporary 'Undiscounted Total' column
df = df.drop(columns=['Undiscounted Total'])


# --- Step 5: Handle missing 'Item' values (categorical) ---
if 'Item' in df.columns:
    num_missing_item_initial = df['Item'].isna().sum()
    if num_missing_item_initial > 0:
        # Fill missing 'Item' based on 'Category' mode
        # Use transform to fill based on group mode, then fill remaining with overall mode
        df['Item'] = df.groupby('Category')['Item'].transform(lambda x: x.mode()[0] if not x.mode().empty else None)

        # Fill any remaining missing 'Item' (e.g., if 'Category' was also missing, or no mode for category) with overall mode
        overall_item_mode = df['Item'].mode()[0] if not df['Item'].mode().empty else 'Unknown Item'
        num_remaining_item_missing = df['Item'].isna().sum()
        if num_remaining_item_missing > 0:
            df.loc[df['Item'].isna(), 'Item'] = overall_item_mode
            missing_values_handled_log['Item'] = missing_values_handled_log.get('Item', []) + [f"{num_missing_item_initial - num_remaining_item_missing} values imputed using category mode, and {num_remaining_item_missing} values imputed with overall mode ({overall_item_mode})."]
        else:
            missing_values_handled_log['Item'] = missing_values_handled_log.get('Item', []) + [f"{num_missing_item_initial} values imputed using category mode."]
    else:
        missing_values_handled_log['Item'] = ['No missing values for Item.']
else:
    missing_values_handled_log['Item'] = ['Item column not found.']


# Verification of missing values after handling
print("\nMissing values count after handling:")
final_missing_counts = df.isnull().sum()
display(final_missing_counts[final_missing_counts > 0])

print("\nMissing values handling summary:")
for col, methods in missing_values_handled_log.items():
    print(f"- {col}:")
    if isinstance(methods, list):
        for method in methods:
            print(f"  - {method}")
    else:
        print(f"  - {methods}")



Missing values count after handling:


,0



Missing values handling summary:
- initial_counts_before_detailed_handling:
  - {}
- Item:
  - No missing values for Item.


### 4. Remove Duplicates

In [48]:
# Identify and count duplicate rows (again, after potential changes from missing value handling)
duplicate_rows_after_cleaning = df.duplicated()
num_duplicate_rows_after_cleaning = duplicate_rows_after_cleaning.sum()

print(f"\nNumber of duplicate rows found before removal: {num_duplicate_rows_after_cleaning}")

# Remove duplicate rows while retaining the first valid occurrence
df.drop_duplicates(inplace=True)

num_removed_duplicates = num_duplicate_rows_after_cleaning - df.duplicated().sum()
print(f"Number of duplicate rows removed: {num_removed_duplicates}")

# Update log
missing_values_handled_log['Duplicates'] = [f"{num_duplicate_rows_after_cleaning} duplicate rows detected, {num_removed_duplicates} removed."]


Number of duplicate rows found before removal: 0
Number of duplicate rows removed: 0


### 5. Clean Data Formatting

In [49]:
# Remove unnecessary leading and trailing spaces from text fields
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
missing_values_handled_log['Formatting'] = missing_values_handled_log.get('Formatting', []) + ['Removed leading/trailing spaces from object columns.']

# Standardize inconsistent capitalization and spelling where appropriate.
# For 'Category', 'Payment Method', 'Location', converting to title case for consistency.
for col in ['Category', 'Payment Method', 'Location', 'Item']:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: x.title() if isinstance(x, str) else x)
missing_values_handled_log['Formatting'].append('Standardized capitalization for Category, Payment Method, Location, and Item columns to Title Case.')

# Ensure numerical columns contain valid numerical values (already handled during missing value imputation)
# Ensure dates are stored in a consistent format
if 'Transaction Date' in df.columns:
    # Convert to datetime, coercing errors will turn invalid dates into NaT
    df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

    # Handle NaT values if any (from invalid date strings)
    num_invalid_dates = df['Transaction Date'].isna().sum()
    if num_invalid_dates > 0:
        # Option 1: Impute with mode/median date. Option 2: Remove rows. Option 3: Fill with a placeholder (e.g., min date)
        # Given instruction to not use generic placeholders and attempt to determine, let's try mode.
        date_mode = df['Transaction Date'].mode()[0]
        df.loc[df['Transaction Date'].isna(), 'Transaction Date'] = date_mode
        missing_values_handled_log['Transaction Date'] = missing_values_handled_log.get('Transaction Date', []) + [f"{num_invalid_dates} invalid date values converted to NaT and then imputed with the mode date ({date_mode.strftime('%Y-%m-%d')})."]
    else:
        missing_values_handled_log['Transaction Date'] = missing_values_handled_log.get('Transaction Date', []) + ['No invalid dates found. Dates formatted to datetime.']

missing_values_handled_log['Formatting'].append('Converted Transaction Date to consistent datetime format.')

print("Data formatting cleanup completed.")

Data formatting cleanup completed.


### 6. Validate the Cleaned Dataset

In [50]:
print("\n--- Final Validation Check ---")

# 1. Verify that all missing values have been appropriately handled.
final_missing_check = df.isnull().sum()
missing_cols_after_cleaning = final_missing_check[final_missing_check > 0]
if missing_cols_after_cleaning.empty:
    print("Validation 1: PASSED - No missing values remain in the dataset.")
else:
    print("Validation 1: FAILED - Missing values still present in columns:")
    display(missing_cols_after_cleaning)
    missing_values_handled_log['Final Missing Check'] = missing_values_handled_log.get('Final Missing Check', []) + ['FAILED - Missing values still present after cleaning.']

# 2. Verify that there are zero duplicate rows.
num_duplicate_rows_final = df.duplicated().sum()
if num_duplicate_rows_final == 0:
    print("Validation 2: PASSED - Zero duplicate rows remain.")
else:
    print(f"Validation 2: FAILED - {num_duplicate_rows_final} duplicate rows still exist.")
    missing_values_handled_log['Final Duplicate Check'] = missing_values_handled_log.get('Final Duplicate Check', []) + [f'FAILED - {num_duplicate_rows_final} duplicate rows remain.']

# 3. Verify that every column has the correct data type.
# Check numerical columns
numerical_cols = ['Price Per Unit', 'Quantity', 'Total Spent']
num_type_check_passed = True
for col in numerical_cols:
    if col in df.columns and not pd.api.types.is_numeric_dtype(df[col]):
        print(f"Validation 3: FAILED - Column '{col}' is not a numeric type (is {df[col].dtype}).")
        num_type_check_passed = False
        missing_values_handled_log['Final Data Type Check'] = missing_values_handled_log.get('Final Data Type Check', []) + [f'FAILED - Column {col} is not numeric.']

# Check 'Discount Applied' for boolean type
if 'Discount Applied' in df.columns and not pd.api.types.is_bool_dtype(df['Discount Applied']):
    print(f"Validation 3: FAILED - Column 'Discount Applied' is not boolean type (is {df['Discount Applied'].dtype}).")
    num_type_check_passed = False
    missing_values_handled_log['Final Data Type Check'] = missing_values_handled_log.get('Final Data Type Check', []) + [f'FAILED - Column Discount Applied is not boolean.']

# Check 'Transaction Date' for datetime type
if 'Transaction Date' in df.columns and not pd.api.types.is_datetime64_any_dtype(df['Transaction Date']):
    print(f"Validation 3: FAILED - Column 'Transaction Date' is not datetime type (is {df['Transaction Date'].dtype}).")
    num_type_check_passed = False
    missing_values_handled_log['Final Data Type Check'] = missing_values_handled_log.get('Final Data Type Check', []) + [f'FAILED - Column Transaction Date is not datetime.']

if num_type_check_passed:
    print("Validation 3: PASSED - Data types appear correct for key columns.")
    missing_values_handled_log['Final Data Type Check'] = missing_values_handled_log.get('Final Data Type Check', []) + ['PASSED - Data types correct.']

# 4. Verify that the 'Discount Applied' column contains ONLY True and False.
if 'Discount Applied' in df.columns:
    unique_discount_values = df['Discount Applied'].unique()
    if set(unique_discount_values).issubset({True, False}):
        print("Validation 4: PASSED - 'Discount Applied' column contains only True/False.")
    else:
        print(f"Validation 4: FAILED - 'Discount Applied' column contains unexpected values: {unique_discount_values}")
        missing_values_handled_log['Final Discount Applied Check'] = missing_values_handled_log.get('Final Discount Applied Check', []) + [f'FAILED - Contains unexpected values: {unique_discount_values}.']
else:
    print("Validation 4: FAILED - 'Discount Applied' column not found.")
    missing_values_handled_log['Final Discount Applied Check'] = missing_values_handled_log.get('Final Discount Applied Check', []) + ['FAILED - Column not found.']

# 5. Check for invalid or inconsistent values again (spot check categorical columns).
# This is already somewhat covered by type checks and title casing, but a final look.
print("Validation 5: Performing spot checks on categorical columns...")
# For simplicity, just display value counts to visually inspect for inconsistencies.
for col in ['Category', 'Payment Method', 'Location']:
    if col in df.columns:
        print(f"  Unique values for '{col}':")
        display(df[col].value_counts())
print("Validation 5: PASSED - Spot checks performed. Manual review of value_counts may be needed for full consistency.")

final_rows, final_cols = df.shape
missing_values_handled_log['Final Dataset Dimensions'] = {'rows': final_rows, 'columns': final_cols}



--- Final Validation Check ---
Validation 1: PASSED - No missing values remain in the dataset.
Validation 2: PASSED - Zero duplicate rows remain.
Validation 3: PASSED - Data types appear correct for key columns.
Validation 4: PASSED - 'Discount Applied' column contains only True/False.
Validation 5: Performing spot checks on categorical columns...
  Unique values for 'Category':


,count
Category,
Electric Household Essentials,1591
Furniture,1591
Food,1588
Milk Products,1584
Butchers,1568
Beverages,1567
Computers And Electric Accessories,1558
Patisserie,1528


  Unique values for 'Payment Method':


,count
Payment Method,
Cash,4310
Digital Wallet,4144
Credit Card,4121


  Unique values for 'Location':


,count
Location,
Online,6354
In-Store,6221


Validation 5: PASSED - Spot checks performed. Manual review of value_counts may be needed for full consistency.


### 7. Export the Dataset

In [51]:
output_filename = 'CLEANED_DATASET.csv'
df.to_csv(output_filename, index=False)
print(f"\nCleaned dataset exported successfully to '{output_filename}'.")
missing_values_handled_log['Export Location'] = output_filename


Cleaned dataset exported successfully to 'CLEANED_DATASET.csv'.


### 8. Provide a Cleaning Summary

In [52]:
print("\n--- Data Cleaning Summary Report ---")
print(f"Original number of rows: {original_rows}")
print(f"Original number of columns: {original_cols}")

print("\nTotal missing values detected (initial):")
for col, count in missing_values_handled_log['initial_counts_before_detailed_handling'].items():
    print(f"- {col}: {count}")

print("\nMissing values found and methods used to replace in each column:")
for col, methods in missing_values_handled_log.items():
    if col not in ['initial_counts_before_detailed_handling', 'Duplicates', 'Formatting', 'Final Missing Check', 'Final Duplicate Check', 'Final Data Type Check', 'Final Discount Applied Check', 'Export Location', 'Final Dataset Dimensions']:
        print(f"- {col}:")
        if isinstance(methods, list):
            for method in methods:
                print(f"  - {method}")
        else:
            print(f"  - {methods}")

print("\nDuplicate Rows Handling:")
if 'Duplicates' in missing_values_handled_log:
    for entry in missing_values_handled_log['Duplicates']:
        print(f"- {entry}")

print("\nOther Corrections Performed (Formatting):")
if 'Formatting' in missing_values_handled_log:
    for entry in missing_values_handled_log['Formatting']:
        print(f"- {entry}")

print("\nFinal Dataset State:")
print(f"Final number of rows: {df.shape[0]}")
print(f"Final number of columns: {df.shape[1]}")

print(f"Confirmation: No duplicate rows remain - {num_duplicate_rows_final == 0}")
print(f"Confirmation: 'Discount Applied' column contains only True and False - {set(df['Discount Applied'].unique()).issubset({True, False})}")
print(f"Location of exported file: {output_filename}")

print("\n--- Cleaning Summary End ---")


--- Data Cleaning Summary Report ---
Original number of rows: 12575
Original number of columns: 11

Total missing values detected (initial):

Missing values found and methods used to replace in each column:
- Item:
  - No missing values for Item.
- Transaction Date:
  - No invalid dates found. Dates formatted to datetime.

Duplicate Rows Handling:
- 0 duplicate rows detected, 0 removed.

Other Corrections Performed (Formatting):
- Removed leading/trailing spaces from object columns.
- Standardized capitalization for Category, Payment Method, Location, and Item columns to Title Case.
- Converted Transaction Date to consistent datetime format.

Final Dataset State:
Final number of rows: 12575
Final number of columns: 11
Confirmation: No duplicate rows remain - True
Confirmation: 'Discount Applied' column contains only True and False - True
Location of exported file: CLEANED_DATASET.csv

--- Cleaning Summary End ---
